# 05 - Batch Normalization & Layer Normalization

*(Notebook bo sung tu bien soan, khong thuoc bo bai tap goc cua Coursera - dung de lap day noi dung "Batch Normalization, Layer Normalization" trong de cuong Chuong 1.)*

Day la phan thu 5 cua bai thuc hanh **P1: Xay dung mo hinh DNN, debug cac van de ve gradient va truc quan hoa gradient flow**. Ba notebook truoc (`01`, `02`, `03`, `04`) da cho thay:
- `03_Khoi_tao_trong_so`: khoi tao trong so kem (zeros / random qua lon) gay ra **vanishing/exploding gradient**.
- Notebook nay tiep tuc cau chuyen do: **Batch Normalization (BN)** va **Layer Normalization (LN)** la hai ky thuat giup on dinh gradient flow **ngay ca khi mang sau va khoi tao khong tot** - mot cach xu ly khac, doc lap voi viec chon initializer.

Khac voi 4 notebook truoc (dang bai tap "dien vao cho trong" cua Coursera), notebook nay la **code hoan chinh, da duoc kiem tra bang gradient checking**, de ban de dang chuyen thanh bai tap cho hoc vien (an mot phan code va yeu cau ho tu viet lai) neu can.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
# Neu chay tren Colab: tu dong git clone repo GitHub cong khai (khong can mount Drive,
# tranh loi "credential propagation was unsuccessful"), roi cd vao dung thu muc cua bai nay,
# de cac lenh 'from xxx_utils import ...' va anh minh hoa trong images/ hoat dong.
import sys, os

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/thực hành/05_Batch-Norm-Layer-Norm"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())

## Muc luc
- [1 - Vi sao can chuan hoa (Normalization)?](#1)
- [2 - Batch Normalization](#2)
- [3 - Layer Normalization](#3)
- [4 - So sanh Batch Norm vs Layer Norm](#4)
- [5 - Debug thuc te: mang sau voi khoi tao kem](#5)
- [6 - Tong ket](#6)

<a name='1'></a>
## 1 - Vi sao can chuan hoa (Normalization)?

Khi lan truyen qua nhieu lop, gia tri pre-activation $Z^{[l]} = W^{[l]}A^{[l-1]} + b^{[l]}$ co the:
- **Lech trung binh va thay doi phuong sai** qua tung layer (cang sau cang khac xa phan phoi ban dau), khien cac layer sau phai lien tuc "thich nghi" voi phan phoi dau vao luon thay doi.
- **Rat lon** (kich hoat bao hoa o sigmoid/tanh -> gradient ~ 0 -> vanishing) hoac **dao dong manh** giua cac layer (exploding), dung nhu ta da thay o notebook `03` voi cac cach khoi tao khac nhau.

**Batch Normalization** (Ioffe & Szegedy, 2015) va **Layer Normalization** (Ba, Kiros & Hinton, 2016) giai quyet van de nay bang mot y tuong chung:

$$\hat{Z} = \frac{Z - \mu}{\sqrt{\sigma^2 + \epsilon}}, \qquad \tilde{Z} = \gamma \odot \hat{Z} + \beta$$

trong do $\gamma, \beta$ la tham so **hoc duoc** (cho phep mang "hoc lai" ty le/do lech neu can, thay vi bi ep buoc ve mean=0, var=1 tuyet doi). Diem khac biet duy nhat giua BN va LN la: **chuan hoa theo truc nao** - do chinh la thu ta se cai dat va so sanh ngay sau day.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bn_ln_utils import sigmoid, relu, relu_backward, load_dataset

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

<a name='2'></a>
## 2 - Batch Normalization

Quy uoc kich thuoc giong cac notebook truoc: $Z$ co shape `(n, m)` - `n` don vi (features/neurons), `m` vi du (examples) theo **cot**.

**BatchNorm chuan hoa theo truc vi du (axis=1)**: voi moi don vi/neuron (moi hang), tinh trung binh va phuong sai **tren toan bo batch** (m vi du), roi chuan hoa. Vi vay BN phu thuoc vao kich thuoc batch, va can xu ly khac nhau giua **train** (dung thong ke cua batch hien tai) va **test** (dung *running mean/var* tich luy tu luc train, vi khi test co the chi co 1 vi du duy nhat).

### Exercise 2.1 - batchnorm_forward

Cong thuc forward (che do train):

$$\mu = \frac{1}{m}\sum_{i=1}^{m} Z_i, \qquad \sigma^2 = \frac{1}{m}\sum_{i=1}^{m} (Z_i - \mu)^2$$
$$\hat{Z} = \frac{Z - \mu}{\sqrt{\sigma^2 + \epsilon}}, \qquad \tilde{Z} = \gamma \hat{Z} + \beta$$

O che do **test**, dung `running_mean`/`running_var` (trung binh truot voi he so `momentum`, cap nhat trong luc train) thay vi thong ke cua batch hien tai.

In [ ]:
def batchnorm_forward(Z, gamma, beta, bn_param, mode):
    """
    Arguments:
    Z -- pre-activation, shape (n, m)
    gamma, beta -- learnable scale/shift, shape (n, 1)
    bn_param -- dict luu 'running_mean', 'running_var', 'momentum', 'eps' (duy tri giua cac lan goi)
    mode -- 'train' hoac 'test'

    Returns:
    out -- Z sau chuan hoa + scale/shift, shape (n, m)
    cache -- cac gia tri can cho backward (None neu mode='test')
    """
    eps = bn_param.get("eps", 1e-8)
    momentum = bn_param.get("momentum", 0.9)
    n, m = Z.shape
    running_mean = bn_param.setdefault("running_mean", np.zeros((n, 1)))
    running_var = bn_param.setdefault("running_var", np.zeros((n, 1)))

    if mode == "train":
        mu = np.mean(Z, axis=1, keepdims=True)
        var = np.var(Z, axis=1, keepdims=True)
        std_inv = 1.0 / np.sqrt(var + eps)
        Z_norm = (Z - mu) * std_inv
        out = gamma * Z_norm + beta

        cache = (Z, Z_norm, mu, var, std_inv, gamma, eps)
        bn_param["running_mean"] = momentum * running_mean + (1 - momentum) * mu
        bn_param["running_var"] = momentum * running_var + (1 - momentum) * var

    elif mode == "test":
        Z_norm = (Z - running_mean) / np.sqrt(running_var + eps)
        out = gamma * Z_norm + beta
        cache = None

    else:
        raise ValueError("mode phai la 'train' hoac 'test'")

    return out, cache

### Exercise 2.2 - batchnorm_backward

Ap dung chain rule qua tung buoc cua forward ( $Z \to \mu,\sigma^2 \to \hat{Z} \to \tilde{Z}$ ), voi `m` la so vi du (batch size):

$$d\beta = \sum_i d\tilde{Z}_i, \qquad d\gamma = \sum_i d\tilde{Z}_i \cdot \hat{Z}_i, \qquad d\hat{Z} = d\tilde{Z} \cdot \gamma$$
$$d\sigma^2 = \sum_i d\hat{Z}_i (Z_i-\mu) \cdot \Big(-\tfrac{1}{2}\Big)(\sigma^2+\epsilon)^{-3/2}$$
$$d\mu = \sum_i d\hat{Z}_i \cdot \Big(\tfrac{-1}{\sqrt{\sigma^2+\epsilon}}\Big) + d\sigma^2 \cdot \text{mean}(-2(Z-\mu))$$
$$dZ = d\hat{Z}\cdot\frac{1}{\sqrt{\sigma^2+\epsilon}} + d\sigma^2\cdot\frac{2(Z-\mu)}{m} + \frac{d\mu}{m}$$

In [ ]:
def batchnorm_backward(dZ_tilde, cache):
    """
    Arguments:
    dZ_tilde -- gradient cua loss theo output cua batchnorm_forward, shape (n, m)
    cache -- tra ve tu batchnorm_forward (mode='train')

    Returns:
    dZ, dgamma, dbeta
    """
    Z, Z_norm, mu, var, std_inv, gamma, eps = cache
    m = Z.shape[1]
    x_mu = Z - mu

    dbeta = np.sum(dZ_tilde, axis=1, keepdims=True)
    dgamma = np.sum(dZ_tilde * Z_norm, axis=1, keepdims=True)

    dZ_norm = dZ_tilde * gamma
    dvar = np.sum(dZ_norm * x_mu, axis=1, keepdims=True) * -0.5 * std_inv**3
    dmu = np.sum(dZ_norm * -std_inv, axis=1, keepdims=True) + dvar * np.mean(-2.0 * x_mu, axis=1, keepdims=True)
    dZ = dZ_norm * std_inv + dvar * 2 * x_mu / m + dmu / m

    return dZ, dgamma, dbeta

### Kiem tra gradient cho BatchNorm

Giong ky thuat da hoc o notebook `02_Kiem_tra_gradient`: so sanh gradient tinh boi `batchnorm_backward` (dao ham giai tich) voi gradient uoc luong bang sai phan huu han (numerical gradient). Neu sai so tuong doi rat nho (~1e-7 tro xuong), backward duoc cai dat dung.

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    """Uoc luong dL/dx bang central difference: (f(x+eps) - f(x-eps)) / (2*eps), tung phan tu cua x."""
    grad = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        orig = x[idx]
        x[idx] = orig + eps
        fp = f()
        x[idx] = orig - eps
        fm = f()
        x[idx] = orig
        grad[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad


def relative_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(1e-8, np.abs(a) + np.abs(b))))

In [ ]:
np.random.seed(0)
n, m = 4, 5
Z = np.random.randn(n, m) * 5
gamma = np.random.randn(n, 1)
beta = np.random.randn(n, 1)
dout = np.random.randn(n, m)  # gia lap gradient tu layer sau truyen ve

out, cache = batchnorm_forward(Z, gamma, beta, {}, mode="train")
dZ, dgamma, dbeta = batchnorm_backward(dout, cache)


def loss():
    o, _ = batchnorm_forward(Z, gamma, beta, {}, mode="train")
    return np.sum(o * dout)   # loss gia lap: <output, dout>  =>  d(loss)/d(output) = dout


dZ_num = numerical_gradient(loss, Z)
dgamma_num = numerical_gradient(loss, gamma)
dbeta_num = numerical_gradient(loss, beta)

print("BatchNorm gradient check:")
print("  dZ     relative error =", relative_error(dZ, dZ_num))
print("  dgamma relative error =", relative_error(dgamma, dgamma_num))
print("  dbeta  relative error =", relative_error(dbeta, dbeta_num))
assert relative_error(dZ, dZ_num) < 1e-6, "batchnorm_backward: dZ sai!"
assert relative_error(dgamma, dgamma_num) < 1e-6, "batchnorm_backward: dgamma sai!"
assert relative_error(dbeta, dbeta_num) < 1e-6, "batchnorm_backward: dbeta sai!"
print("Tat ca deu dung (sai so < 1e-6).")

<a name='3'></a>
## 3 - Layer Normalization

**LayerNorm chuan hoa theo truc feature (axis=0)**: voi **moi vi du** (moi cot), tinh trung binh/phuong sai **tren toan bo cac feature cua chinh no**, khong lien quan gi den cac vi du khac trong batch.

He qua quan trong:
- LN **khong phu thuoc kich thuoc batch** - ngay ca batch_size=1 van tinh duoc thong ke hop ly. Vi vay LN **khong can phan biet train/test**, khong can running mean/var.
- Day la ly do LN duoc dung pho bien trong RNN/Transformer (chuoi co do dai/kich thuoc batch thay doi, thong ke theo batch kem tin cay), trong khi BN pho bien hon trong CNN/MLP voi batch size du lon va on dinh.

Cong thuc gan nhu giong het BN, chi doi truc tinh $\mu, \sigma^2$ tu `axis=1` (theo vi du) sang `axis=0` (theo feature):

In [ ]:
def layernorm_forward(Z, gamma, beta, eps=1e-8):
    """
    Giong batchnorm_forward, nhung mu/var tinh theo axis=0 (tren cac feature, rieng cho tung vi du)
    thay vi axis=1 (tren cac vi du). Khong can phan biet train/test.
    """
    mu = np.mean(Z, axis=0, keepdims=True)
    var = np.var(Z, axis=0, keepdims=True)
    std_inv = 1.0 / np.sqrt(var + eps)
    Z_norm = (Z - mu) * std_inv
    out = gamma * Z_norm + beta

    cache = (Z, Z_norm, mu, var, std_inv, gamma, eps)
    return out, cache


def layernorm_backward(dZ_tilde, cache):
    """Cung cong thuc voi batchnorm_backward, chi doi axis=1 -> axis=0 va m -> n (so feature)."""
    Z, Z_norm, mu, var, std_inv, gamma, eps = cache
    n = Z.shape[0]
    x_mu = Z - mu

    dbeta = np.sum(dZ_tilde, axis=1, keepdims=True)
    dgamma = np.sum(dZ_tilde * Z_norm, axis=1, keepdims=True)

    dZ_norm = dZ_tilde * gamma
    dvar = np.sum(dZ_norm * x_mu, axis=0, keepdims=True) * -0.5 * std_inv**3
    dmu = np.sum(dZ_norm * -std_inv, axis=0, keepdims=True) + dvar * np.mean(-2.0 * x_mu, axis=0, keepdims=True)
    dZ = dZ_norm * std_inv + dvar * 2 * x_mu / n + dmu / n

    return dZ, dgamma, dbeta

**Luu y:** `dgamma`/`dbeta` van cong don theo `axis=1` (tren cac vi du) o ca hai ham, vi $\gamma,\beta$ la tham so **theo tung feature**, dung chung cho moi vi du trong batch - chi thong ke $\mu,\sigma^2$ (dac trung cho BN hay LN) la khac truc.

Kiem tra gradient tuong tu nhu voi BatchNorm:

In [ ]:
np.random.seed(1)
n, m = 4, 5
Z = np.random.randn(n, m) * 5
gamma = np.random.randn(n, 1)
beta = np.random.randn(n, 1)
dout = np.random.randn(n, m)

out, cache = layernorm_forward(Z, gamma, beta)
dZ, dgamma, dbeta = layernorm_backward(dout, cache)


def loss():
    o, _ = layernorm_forward(Z, gamma, beta)
    return np.sum(o * dout)


dZ_num = numerical_gradient(loss, Z)
dgamma_num = numerical_gradient(loss, gamma)
dbeta_num = numerical_gradient(loss, beta)

print("LayerNorm gradient check:")
print("  dZ     relative error =", relative_error(dZ, dZ_num))
print("  dgamma relative error =", relative_error(dgamma, dgamma_num))
print("  dbeta  relative error =", relative_error(dbeta, dbeta_num))
assert relative_error(dZ, dZ_num) < 1e-6, "layernorm_backward: dZ sai!"
assert relative_error(dgamma, dgamma_num) < 1e-6, "layernorm_backward: dgamma sai!"
assert relative_error(dbeta, dbeta_num) < 1e-6, "layernorm_backward: dbeta sai!"
print("Tat ca deu dung (sai so < 1e-6).")

<a name='4'></a>
## 4 - So sanh Batch Norm vs Layer Norm

| Tieu chi | Batch Normalization | Layer Normalization |
|---|---|---|
| Truc chuan hoa | Theo **vi du** (axis=1) - moi feature chuan hoa tren ca batch | Theo **feature** (axis=0) - moi vi du chuan hoa tren cac feature cua no |
| Phu thuoc batch size | Co - can batch du lon de thong ke on dinh | Khong - hoat dong tot voi batch size = 1 |
| Train vs Test | Khac nhau (test dung running mean/var) | Giong nhau (khong can running stats) |
| Hay dung o | MLP, CNN (batch size lon va co dinh) | RNN, Transformer, batch size nho/thay doi |
| Tham so hoc duoc | $\gamma, \beta$ theo feature (giong nhau ca hai) | $\gamma, \beta$ theo feature (giong nhau ca hai) |

Ca hai deu giai quyet cung mot van de (chuan hoa phan phoi cua $Z$ de gradient khong bi vanishing/exploding khi lan truyen qua nhieu layer) - chi khac **truc** thuc hien phep chuan hoa.

<a name='5'></a>
## 5 - Debug thuc te: mang sau voi khoi tao kem

Quay lai dung tinh huong nhu notebook `03_Khoi_tao_trong_so`: dung **khoi tao trong so kem** (`W *= 4`, tuong tu "random initialization" da lam mang that bai o notebook 03), nhung lan nay tren mot mang **sau hon** (4 hidden layer thay vi 2), va so sanh 3 phuong an:
- `none`: khong chuan hoa gi ca (baseline - se that bai, dung nhu du doan)
- `batchnorm`: chen BatchNorm sau moi linear layer (truoc ReLU)
- `layernorm`: chen LayerNorm sau moi linear layer (truoc ReLU)

Kien truc: `LINEAR -> NORM? -> RELU` x4 -> `LINEAR -> SIGMOID`, dung lai chinh dataset "circles" cua notebook `03`.

In [ ]:
def initialize_parameters(layers_dims, init_type="random_bad"):
    """
    init_type='random_bad': W *= 4 (co tinh, de tao vanishing/exploding gradient ro ret - xem notebook 03)
    init_type='he': He initialization (khoi tao tot)
    Voi moi hidden layer, them gamma=1, beta=0 (dung khi co chuan hoa; vo hai neu norm_type='none').
    """
    np.random.seed(3)
    parameters = {}
    L = len(layers_dims) - 1
    for l in range(1, L + 1):
        scale = 4 if init_type == "random_bad" else np.sqrt(2.0 / layers_dims[l - 1])
        parameters[f"W{l}"] = np.random.randn(layers_dims[l], layers_dims[l - 1]) * scale
        parameters[f"b{l}"] = np.zeros((layers_dims[l], 1))
        if l < L:  # khong chuan hoa layer output
            parameters[f"gamma{l}"] = np.ones((layers_dims[l], 1))
            parameters[f"beta{l}"] = np.zeros((layers_dims[l], 1))
    return parameters

In [ ]:
def forward_propagation(X, parameters, layers_dims, norm_type, bn_states, mode="train"):
    """
    norm_type: 'none' | 'batchnorm' | 'layernorm'
    bn_states: dict {layer_index: bn_param} - chi dung cho batchnorm (luu running mean/var)
    """
    L = len(layers_dims) - 1
    A_prev = X
    caches = {}

    for l in range(1, L):
        W, b = parameters[f"W{l}"], parameters[f"b{l}"]
        Z = np.dot(W, A_prev) + b

        norm_cache = None
        Z_tilde = Z
        if norm_type == "batchnorm":
            gamma, beta = parameters[f"gamma{l}"], parameters[f"beta{l}"]
            Z_tilde, norm_cache = batchnorm_forward(Z, gamma, beta, bn_states[l], mode)
        elif norm_type == "layernorm":
            gamma, beta = parameters[f"gamma{l}"], parameters[f"beta{l}"]
            Z_tilde, norm_cache = layernorm_forward(Z, gamma, beta)

        A = relu(Z_tilde)
        caches[l] = (A_prev, W, b, Z_tilde, norm_cache, A)
        A_prev = A

    WL, bL = parameters[f"W{L}"], parameters[f"b{L}"]
    ZL = np.dot(WL, A_prev) + bL
    AL = sigmoid(ZL)
    caches[L] = (A_prev, WL, bL, ZL, None, AL)

    return AL, caches


def compute_cost(AL, Y):
    m = Y.shape[1]
    logprobs = np.multiply(-np.log(AL), Y) + np.multiply(-np.log(1 - AL), 1 - Y)
    return (1.0 / m) * np.nansum(logprobs)

In [ ]:
def backward_propagation(X, Y, caches, layers_dims, norm_type):
    L = len(layers_dims) - 1
    m = X.shape[1]
    grads = {}

    A_prev, WL, bL, ZL, _, AL = caches[L]
    dZ = (1.0 / m) * (AL - Y)
    grads[f"dW{L}"] = np.dot(dZ, A_prev.T)
    grads[f"db{L}"] = np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(WL.T, dZ)

    for l in reversed(range(1, L)):
        A_prev, W, b, Z_tilde, norm_cache, A = caches[l]
        dZ_tilde = relu_backward(dA_prev, Z_tilde)

        if norm_type == "batchnorm":
            dZ, dgamma, dbeta = batchnorm_backward(dZ_tilde, norm_cache)
            grads[f"dgamma{l}"], grads[f"dbeta{l}"] = dgamma, dbeta
        elif norm_type == "layernorm":
            dZ, dgamma, dbeta = layernorm_backward(dZ_tilde, norm_cache)
            grads[f"dgamma{l}"], grads[f"dbeta{l}"] = dgamma, dbeta
        else:
            dZ = dZ_tilde

        grads[f"dW{l}"] = np.dot(dZ, A_prev.T)
        grads[f"db{l}"] = np.sum(dZ, axis=1, keepdims=True)
        dA_prev = np.dot(W.T, dZ)

    return grads


def update_parameters(parameters, grads, learning_rate, layers_dims, norm_type):
    L = len(layers_dims) - 1
    for l in range(1, L + 1):
        parameters[f"W{l}"] -= learning_rate * grads[f"dW{l}"]
        parameters[f"b{l}"] -= learning_rate * grads[f"db{l}"]
        if norm_type in ("batchnorm", "layernorm") and l < L:
            parameters[f"gamma{l}"] -= learning_rate * grads[f"dgamma{l}"]
            parameters[f"beta{l}"] -= learning_rate * grads[f"dbeta{l}"]
    return parameters

### model(): huan luyen + theo doi gradient flow

Giong ky thuat "gradient flow visualization" da lam o notebook `03`: ghi lai `||dW||` cua tung layer moi 10 iteration, de ve bieu do gradient flow sau khi train xong.

In [ ]:
def model(X, Y, layers_dims, norm_type="none", init_type="random_bad",
          learning_rate=0.1, num_iterations=3000, track_every=10):
    parameters = initialize_parameters(layers_dims, init_type)
    L = len(layers_dims) - 1
    bn_states = {l: {} for l in range(1, L)}  # chi thuc su dung khi norm_type='batchnorm'

    grad_norms = {f"dW{l}": [] for l in range(1, L + 1)}
    tracked_iters, costs = [], []

    for i in range(num_iterations):
        AL, caches = forward_propagation(X, parameters, layers_dims, norm_type, bn_states, mode="train")
        cost = compute_cost(AL, Y)
        grads = backward_propagation(X, Y, caches, layers_dims, norm_type)
        parameters = update_parameters(parameters, grads, learning_rate, layers_dims, norm_type)

        if i % track_every == 0:
            tracked_iters.append(i)
            costs.append(cost)
            for l in range(1, L + 1):
                grad_norms[f"dW{l}"].append(np.linalg.norm(grads[f"dW{l}"]))

    return parameters, bn_states, costs, grad_norms, tracked_iters

In [ ]:
train_X, train_Y = load_dataset()
layers_dims = [2, 20, 20, 20, 20, 1]  # 4 hidden layer + 1 output -> du sau de vanishing/exploding the hien ro

histories = {}
for norm_type in ["none", "batchnorm", "layernorm"]:
    parameters, bn_states, costs, grad_norms, tracked_iters = model(
        train_X, train_Y, layers_dims, norm_type=norm_type, init_type="random_bad",
        learning_rate=0.1, num_iterations=3000, track_every=10,
    )
    histories[norm_type] = {
        "parameters": parameters, "bn_states": bn_states,
        "costs": costs, "grad_norms": grad_norms, "iters": tracked_iters,
    }
    print(f"norm_type = {norm_type:10s} -> final cost = {costs[-1]:.4f}")

### 5.1 - Bieu do gradient flow: `none` vs `batchnorm` vs `layernorm`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
colors = plt.cm.viridis(np.linspace(0, 1, 4))

for ax, norm_type in zip(axes, ["none", "batchnorm", "layernorm"]):
    h = histories[norm_type]
    for l in range(1, 5):
        ax.plot(h["iters"], h["grad_norms"][f"dW{l}"], label=f"dW{l}", color=colors[l - 1])
    ax.set_yscale("symlog", linthresh=1e-8)
    ax.set_title(f'Gradient flow - norm_type = "{norm_type}"')
    ax.set_xlabel("iteration")
    ax.set_ylabel("||dW|| (symlog)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Doc bieu do:**
- **`none`**: `||dW||` tang vot (~10^4) ngay iteration dau (exploding, do `W*=4` qua 5 layer khuech dai tin hieu manh) roi lap tuc roi ve xap xi 0 va nam yen o do het qua trinh train (cac neuron ReLU "chet" - dead ReLU vi pre-activation am het). Dung mo hinh khong hoc duoc gi (cost dung yen o $\ln 2 \approx 0.693$, tuong duong doan ngau nhien).
- **`batchnorm`**: `||dW||` on dinh trong mot dai hop ly (~1e-3 den 1e-1) suot qua trinh train, khong bung no, khong bien mat.
- **`layernorm`**: cung on dinh, khong exploding/vanishing, tuy dong luc hoi khac (giam roi tang nhe khi mo hinh bat dau hoc duoc nhieu hon).

=> Ca BN va LN deu **"cuu" duoc gradient flow** cho mang sau du khoi tao rat kem - day la bang chung truc quan cho vai tro cua normalization ben canh weight initialization (notebook 03) trong viec xu ly vanishing/exploding gradient.

### 5.2 - Duong cost & decision boundary

In [ ]:
plt.figure(figsize=(7, 4.5))
for norm_type in ["none", "batchnorm", "layernorm"]:
    plt.plot(histories[norm_type]["iters"], histories[norm_type]["costs"], label=norm_type)
plt.xlabel("iteration")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def predict_dec(X, parameters, layers_dims, norm_type, bn_states):
    AL, _ = forward_propagation(X, parameters, layers_dims, norm_type, bn_states, mode="test")
    return AL > 0.5


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_min, x_max = train_X[0, :].min() - 1, train_X[0, :].max() + 1
y_min, y_max = train_X[1, :].min() - 1, train_X[1, :].max() + 1
h = 0.02
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()].T

for ax, norm_type in zip(axes, ["none", "batchnorm", "layernorm"]):
    h_ = histories[norm_type]
    Z = predict_dec(grid, h_["parameters"], layers_dims, norm_type, h_["bn_states"])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    ax.scatter(train_X[0, :], train_X[1, :], c=train_Y, cmap=plt.cm.Spectral, edgecolors="k", s=15)
    ax.set_title(f'norm_type = "{norm_type}"  (cost={h_["costs"][-1]:.3f})')

plt.tight_layout()
plt.show()

**Nhan xet:** `none` khong tach duoc lop trong/ngoai nao ca (to mau dong nhat toan bo mat phang - dung nhu du doan tu cost khong giam). Ca `batchnorm` va `layernorm` deu ve duoc duong bien phan tach gan dung voi hinh vanh khan cua du lieu - `layernorm` trong vi du nay cho duong bien muot hon, nhung ket qua cu the phu thuoc learning rate/so iteration; diem mau chot can nho la **ca hai deu hoc duoc**, trong khi `none` thi khong.

<a name='6'></a>
## 6 - Tong ket

<font color='blue'>

**Nhung gi ban nen nho tu notebook nay:**
- BatchNorm va LayerNorm cung giai quyet van de gradient vanishing/exploding nhu weight initialization (notebook 03), nhung theo huong khac: thay vi chi chon $W$ ban dau that can than, chung **chu dong dieu chinh phan phoi cua $Z$ o moi buoc forward**, giup gradient flow on dinh **xuyen suot qua trinh train**, khong chi luc khoi tao.
- Khac biet duy nhat giua BN va LN: **truc tinh thong ke** ($\mu,\sigma^2$) - theo vi du (BN) hay theo feature (LN). Dieu nay dan den khac biet quan trong ve su phu thuoc batch size va cach xu ly train/test.
- Ky thuat **gradient checking** (notebook 02) va **gradient flow visualization** (notebook 03) deu tai su dung duoc o day - day la cac cong cu debug tong quat, ap dung cho bat ky thanh phan moi nao ban them vao mang, khong rieng gi vi du trong bai nay.

**Goi y mo rong (neu ban muon giao bai tap cho hoc vien):** an di phan code trong `batchnorm_backward`/`layernorm_backward` (Exercise 2.2 va phan tuong ung o muc 3), yeu cau hoc vien tu suy ra va cai dat, roi dung gradient check co san de tu cham diem.